# Lab 02 · Prompt caching, and how to destroy it
**~25 minutes · costs about $0.05 · Domain 2 API Mechanics, Domain 5 Cost**

The single highest-value lab here. Nearly every cost question on the exam
resolves to one fact: **caching is an exact prefix match**. You will see your
own hit rate collapse because of one interpolated value, and you will not
forget it afterwards.

In [ ]:
import os, anthropic
client = anthropic.Anthropic()          # reads ANTHROPIC_API_KEY
MODEL  = "claude-sonnet-4-6"            # verify against lab 00 output
CHEAP  = "claude-haiku-4-5"             # for high-volume steps
print("sdk", anthropic.__version__)

## Build a prefix big enough to cache

Minimum cacheable granularity is roughly 1,024 tokens on the larger models, so
a two-line system prompt will silently never cache. That alone catches people.

In [ ]:
DOC = ("Section %d. The widget subsystem handles inventory reconciliation "
       "across regional warehouses, applying tier rules and audit logging.\n")
CORPUS = "".join(DOC % i for i in range(1, 160))
print("approx tokens:",
      client.messages.count_tokens(model=MODEL,
          messages=[{"role":"user","content":CORPUS}]).input_tokens)

## Run 1 · cache write

The first call pays a write premium (1.25x input at the 5-minute TTL).

In [ ]:
def ask(question, prefix):
    r = client.messages.create(
        model=MODEL, max_tokens=100,
        system=[
            {"type":"text","text":"You answer from the provided corpus only."},
            {"type":"text","text":prefix,"cache_control":{"type":"ephemeral"}},
        ],
        messages=[{"role":"user","content":question}],
    )
    u = r.usage
    print(f"  write={u.cache_creation_input_tokens:<7} read={u.cache_read_input_tokens:<7} "
          f"uncached_in={u.input_tokens}")
    return r

print("run 1"); ask("What does the widget subsystem do?", CORPUS);

## Run 2 · cache read

Same prefix. `write` should drop to 0 and `read` should jump.

In [ ]:
print("run 2"); ask("Which rules does it apply?", CORPUS);
print("run 3"); ask("Does it log?", CORPUS);

## Now break it

One timestamp at the top of the prefix. Nothing else changes.

This is the exam scenario: *"a release added a current timestamp to the cached
system prompt and `cache_read_input_tokens` dropped to near zero."*

In [ ]:
from datetime import datetime
print("poisoned run 1"); ask("What does it do?", f"Generated: {datetime.now()}\n" + CORPUS);
print("poisoned run 2"); ask("Which rules?",     f"Generated: {datetime.now()}\n" + CORPUS);
print()
print("read stays at 0. Every call is now a fresh write at 1.25x input price.")
print("The fix is PLACEMENT, not TTL: move volatile values out of the cached prefix.")

## Do the arithmetic

Cache reads cost about 0.1x base input. Work out what the mistake costs at your
real traffic volume.

In [ ]:
IN_PER_MTOK, CACHE_READ_MULT, CACHE_WRITE_MULT = 3.00, 0.10, 1.25
tokens, calls = 5000, 100_000

full  = tokens/1e6 * IN_PER_MTOK * calls
cached = (tokens/1e6*IN_PER_MTOK*CACHE_WRITE_MULT) + (tokens/1e6*IN_PER_MTOK*CACHE_READ_MULT*(calls-1))
print(f"no cache      ${full:,.2f}")
print(f"cached        ${cached:,.2f}")
print(f"saved         ${full-cached:,.2f}  ({(1-cached/full)*100:.1f}%)")

---
### Checkpoint
- What is the prefix order? (three things, in order)
- Name three ways to invalidate a cache
- Where do you place breakpoints, and why does content order matter?